# 5.3 Your turn: deaths follow tests — but for how long?

Every relationship in this lesson so far was checked by asking what would make it
believable: a mechanism, evidence at the unit of the claim, and a place you have not
looked yet ([05.1](05.1-relationships.ipynb)). This notebook takes one relationship with a
mechanism nobody doubts and asks the question that usually gets skipped: **is it stable
enough to write down as a model?**

The relationship: in the Dutch COVID figures, deaths follow positive tests. Someone who
tests positive and dies of it does so about two weeks later, so the deaths recorded
fourteen days from now are the deaths *of* today's positive tests. That is a lag, and once
the lag is taken out the two series should move together — as long as the fraction of
positive tests that ends in death stays the same.

That last clause is the claim, and it is exactly the kind a straight line makes: one ratio
of deaths to tests, the same on every day. This notebook fits that line, tests it the way
[04.2](../lesson4/04.2-distribution_fitting.ipynb) tests every model — on the residual —
and finds a window where the line is the whole story and a window where it is not. The
last section hands the broken model to you.

In [ ]:
import numpy as np
from goad_toolkit.analytics import DistributionFitter, fit_table
from goad_toolkit.config import DataConfig, FileConfig
from goad_toolkit.dataprocessor import CovidDataProcessor
from goad_toolkit.models import linear_model, mse, train_model
from goad_toolkit.visualizer import (
    ComparePlot,
    ComparePlotDate,
    FitPlotSettings,
    PlotFits,
    PlotSettings,
    ResidualPlot,
)
from loguru import logger

## 5.3.1 The data, and what the pipeline decides

Daily Dutch COVID figures, October 2020 to June 2021: positive tests and deaths per day,
downloaded once and cached. `CovidDataProcessor` is a `Pipeline` of the kind lesson 1
built, and every step in it is a decision about what the data *means* rather than a
cleaning chore:

1. `DiffValues` on `deaths` — the source reports the cumulative count; the difference is
   the number of deaths that day.
2. `ShiftValues(period=-14)` → `deaths_shifted` — the deaths recorded fourteen days
   *later* are moved back onto today's row, next to the tests that led to them. A model
   between the two needs this lag; without it the pairs have nothing to do with each other.
3. `SelectDataRange` — the window.
4. `RollingAvg(window=7)` on both — reporting has a weekly cycle (fewer registrations at
   weekends) that is not part of the disease.
5. `ZScaler` → `*_zscore` — deaths are tens a day, tests are thousands. To see whether the
   two series have the same *shape*, put them on one scale.

The lag is the one to hold on to. Fourteen days is itself a claim about the mechanism —
infection, illness, death — and `DataConfig(period=-14)` is where that claim lives.

In [ ]:
processor = CovidDataProcessor(FileConfig(), DataConfig())
data = processor.process()
print(f"{len(data)} days, {data.index.min().date()} to {data.index.max().date()}")
data[["positivetests", "deaths", "deaths_shifted", "positivetests_zscore", "deaths_shifted_zscore"]].head(3)

In [ ]:
fig, ax = ComparePlot(PlotSettings(figsize=(11, 4), xlabel="date", ylabel="z-score",
                                   title="Positive tests and (lagged) deaths, on one scale")).plot(
    data=data, x="date", y1="positivetests_zscore", y2="deaths_shifted_zscore",
)

Read it as two halves. Through the autumn and winter the curves sit on top of each other:
when tests rise, deaths two weeks later rise by a proportional amount, and when tests fall,
so do deaths. From March the two part company — tests climb again and deaths do not
follow. Keep that in mind, but do not fit to it yet. The honest way to find out whether one
ratio describes this relationship is to fit the line where it plainly holds, and then ask
the residual what is left.

## 5.3.2 A line, on the window where the ratio looks constant

The Dutch vaccination programme started on 6 January 2021. Fit the line on the days before
it: `deaths_shifted ≈ a · positivetests + b`, with goad's modelling kit — `linear_model` is
the shape, `mse` is the loss, `train_model` finds `a` and `b`. The bounds say what you
already know: a ratio of deaths to tests is not negative, and neither is the intercept.

In [ ]:
VACCINATION_START = "2021-01-06"
before = data.index < VACCINATION_START

tests = data["positivetests"].to_numpy()
deaths = data["deaths_shifted"].to_numpy()

params_before = train_model(tests[before], deaths[before], linear_model, mse, [0.01, 1.0],
                            bounds=[(0, 1.0), (0, None)])
print(f"deaths ≈ {params_before[0]:.4f} · tests + {params_before[1]:.1f}")

early = data[before].copy()
early["predicted"] = linear_model(tests[before], params_before)
early["residual"] = early["deaths_shifted"] - early["predicted"]

fig, ax = ComparePlot(PlotSettings(figsize=(11, 4), xlabel="date", ylabel="deaths",
                                   title="Before vaccination: deaths as a straight line in positive tests")).plot(
    data=early, x="date", y1="deaths_shifted", y2="predicted",
)

The line follows the winter wave up and over the peak and down again. A fit that looks
good is where most analyses stop, and it is exactly the wrong place to stop: the plot shows
what the model got, and the question is what it missed. Subtract the line and look at the
error, day by day.

In [ ]:
fig, ax = ResidualPlot(PlotSettings(figsize=(11, 3.5), xlabel="date", ylabel="error",  # ty: ignore[invalid-argument-type]
                                    title="Before vaccination: what the line did not explain")).plot(
    data=early, x="date", y="residual", date=VACCINATION_START, datelabel="vaccination starts",
    interval=2,
)

Waves of a week or two either side of zero — the size a 7-day smoothing of noisy daily
counts leaves behind — and no drift across the window. That is the first check on a
residual, **no structure in time**, and it passes. The second check is the distribution of
the error itself, which is what `DistributionFitter` is for inside a modelling loop: fit
every continuous family and read the table for agreement, not for a winner.

In [ ]:
fitter = DistributionFitter(seed=42)
fits_before = fitter.fit(early["residual"].to_numpy(), discrete=False)
fit_table(fits_before)[["distribution", "log_likelihood", "ks_pvalue", "best_likelihood", "best_ks"]].head(6)

In [ ]:
fig = PlotFits(PlotSettings(figsize=(12, 4), xlabel="error", ylabel="density",
                            title="Before vaccination: the residual, top three fits")).plot(
    data=early["residual"].to_numpy(), fit_results=fits_before,
    fitplotsettings=FitPlotSettings(bins=25, max_fits=3),
)

Centred on zero, and half a dozen families within a few log-likelihood points of each
other — beta, weibull, gamma, lognormal, skew-normal — none of them rejected by KS. The one
family that *is* rejected, `uniform`, is rejected for the right reason: the errors cluster
near zero rather than spreading evenly. When the winner is not meaningfully ahead, the data
has not distinguished the families — and that is what noise looks like.

Say it explicitly, because it is the bar the rest of this notebook holds every model to:
**a symmetric residual from a plausible family, with no pattern over time, is what "this
model is done" looks like.** Not a perfect fit — the weekly waves are still there — but
nothing left that has a *shape*, and therefore nothing left that names a mechanism the
model lacks. On this window, one ratio of deaths to tests is very nearly the whole story.

## 5.3.3 The same line, on the full window

Now take the model exactly as fitted — the same `a`, the same `b`, nothing re-estimated —
and run it over every day in the window. Nothing in it knows what month it is, so if the
ratio really is one number, the residual should go on looking like the plot above.

In [ ]:
data["predicted"] = linear_model(tests, params_before)
data["residual"] = data["deaths_shifted"] - data["predicted"]
print(f"mse before vaccination {mse(deaths[before], data.loc[before, 'predicted']):.0f}, "
      f"after {mse(deaths[~before], data.loc[~before, 'predicted']):.0f}")

fig, ax = ComparePlotDate(PlotSettings(figsize=(11, 4), xlabel="date", ylabel="deaths",
                                       title="The pre-vaccination line, run over the full window")).plot(
    data=data, x="date", y1="deaths_shifted", y2="predicted",
    date=VACCINATION_START, datelabel="vaccination starts",
)

In [ ]:
fig, ax = ResidualPlot(PlotSettings(figsize=(11, 4), xlabel="date", ylabel="error",
                                    title="The full window: a residual with a shape")).plot(
    data=data, x="date", y="residual", date=VACCINATION_START,
    datelabel="vaccination starts", interval=1,
)

In [ ]:
fits_full = fitter.fit(data["residual"].to_numpy(), discrete=False)
fit_table(fits_full)[["distribution", "log_likelihood", "ks_pvalue", "best_likelihood", "best_ks"]].head(4)

Let the plot make the argument. Left of the line, the residual is the one you already
accepted as noise. Right of it, the errors slide below zero and stay there: from March on,
month after month, the line predicts more deaths than happen, and the run never comes back. The fit table says the same thing in its own language — every family rejected,
and `uniform` "winning", which is what a fitter reports when the errors are spread along a
trend rather than scattered around a centre.

That is not a distribution problem, and it is not a fitting problem either: the same line
was noise a month earlier. **The relationship changed.** After January, the same number of
positive tests produces fewer deaths, and a model with one fixed ratio has no way to say
so. The residual just did — and it also said roughly when.

## 5.3.4 Your turn: name the shape, add the term

Two responses to a residual like that are tempting, and both are wrong.

The first is to make the model much more complex — more parameters, a polynomial in time,
a separate line per month — until the residual is flat by force. That matches the picture
and explains nothing, and lesson 6 shows what it does to prediction.

The second is to drop vaccination from the story: the ratio changed, the reasons are
complicated, report the correlation and move on. That throws away the one thing the
residual handed you, which is *when* the change began.

The problem is very doable if you spend a little longer thinking about what the data
means. You know three things. Vaccination started on 6 January 2021. The rollout began
with the people most likely to die of the disease. So the fraction of positive tests that
ends in death should *fall* as the rollout proceeds — not on one day, but over the weeks
it takes to reach them.

So the question is not "which model". It is: **what does the ratio of deaths to tests do
over time?** One value, then a transition, then another value — what shape is that? Look
at the residual again, or compute the ratio directly and plot it. Then write that shape
into the model as a term, next to the line you already have, with the day number as its
input. `goad_toolkit.models.logistic` is one of the shapes goad ships; whether it is the
right one, and how it should combine with the line — added to it, or multiplied into it —
is the decision that *is* the model. Decide it from what the claim means, not from which
version fits better.

Three things to settle before you train anything:

1. **The shape**, in one sentence about the ratio, before you touch any function.
2. **How it combines with the line.** "Some fixed number of deaths a day disappears" and
   "the ratio of deaths to tests goes down" are different claims, and they are different
   formulas.
3. **Starting values and bounds.** `a` and `b` you already have. For the term you add, the
   z-score plot tells you roughly when the turn happens and which way it goes; an optimiser
   started nowhere near the answer finds a different one, and bounds are how you say what a
   parameter is allowed to mean.

Work in the cells below. The loop is already wired up, so once `covid_model` does something,
the rest runs. The worked answer is in [05.4](05.4-covid-solution.ipynb) — read it after
your own residual is flat, not before.

In [ ]:
# >>> Your turn: look at the ratio itself. What does deaths_shifted / positivetests do over time? <<<

### Model

The model function takes two inputs per day — the positive tests and the day number,
stacked into one array — and a list of parameters. The version below is the straight line
and nothing else: it uses the first two parameters and ignores the rest, which is exactly
the model that just failed. Replace it with yours, and give `initial_params` and `bounds`
an entry for every parameter you add.

In [ ]:
day = np.arange(len(data)).astype(float)
X = np.stack([tests, day], axis=1)
vaccination_day = float(np.argmax(data.index >= VACCINATION_START))
print(f"X has shape {X.shape}: tests in column 0, day number in column 1; vaccination starts on day {vaccination_day:.0f}")


def covid_model(X: np.ndarray, params: list[float]) -> np.ndarray:
    """>>> Your turn: the line, combined with the shape you named. <<<"""
    a, b = params[:2]
    return linear_model(X[:, 0], [a, b])


initial_params = [params_before[0], params_before[1]]  # >>> add starting values for your own term <<<
bounds = [(0, 1.0), (0, None)]  # >>> and what each parameter is allowed to mean <<<

### Train

`train_model` minimises the loss over the parameters, from your starting values, inside
your bounds. If it fails, the usual reason is that `params` has a different length from
what `covid_model` unpacks.

In [ ]:
try:
    params = train_model(X, deaths, covid_model, mse, initial_params, bounds=bounds)
    logger.success(f"fitted parameters: {np.round(params, 4)}")
except Exception as error:
    logger.error(f"training failed: {error}")
    params = initial_params

### Predict

The fit against the data, with the vaccination start marked — and then the residual, which
is the plot that decides whether you are done.

In [ ]:
data["your model"] = covid_model(X, params)
data["your residual"] = data["deaths_shifted"] - data["your model"]
print(f"mse: the pre-vaccination line on the full window {mse(deaths, data['predicted']):.0f}  →  "
      f"your model {mse(deaths, data['your model']):.0f}")

fig, ax = ComparePlotDate(PlotSettings(figsize=(11, 4), xlabel="date", ylabel="deaths",
                                       title="Deaths, and your model")).plot(
    data=data, x="date", y1="deaths_shifted", y2="your model",
    date=VACCINATION_START, datelabel="vaccination starts",
)

In [ ]:
fig, ax = ResidualPlot(PlotSettings(figsize=(11, 4), xlabel="date", ylabel="error",
                                    title="What your model did not explain")).plot(
    data=data, x="date", y="your residual", date=VACCINATION_START,
    datelabel="vaccination starts", interval=1,
)

### Test the residual

The same two checks as §5.3.2, against the same bar: no structure over time, and a
symmetric distribution that several families fit about equally well. If the long negative
run is still there, the term you added is not saying what you meant it to say — go back to
the shape, not to the optimiser.

In [ ]:
fits_yours = fitter.fit(data["your residual"].to_numpy(), discrete=False)
print(fit_table(fits_yours)[["distribution", "log_likelihood", "ks_pvalue", "best_likelihood", "best_ks"]].head(4).to_string())

fig = PlotFits(PlotSettings(figsize=(12, 4), xlabel="error", ylabel="density",
                            title="Your residual, top three fits")).plot(
    data=data["your residual"].to_numpy(), fit_results=fits_yours,
    fitplotsettings=FitPlotSettings(bins=30, max_fits=3),
)

## 5.3.5 What to write down

1. **The shape, in words, and the formula it became.** Why that combination with the line
   and not the other one — the sentence about the ratio is the justification.
2. **What the fitted parameters say.** Where the turn is halfway, how steep it is, and how
   that compares with the vaccination rollout. That comparison is the finding.
3. **What the model does not say.** It has a term on the calendar, not a term for
   vaccination. The end of the winter wave, who was being tested, and the order in which
   age groups were vaccinated all sit on the same calendar. Which of 05.1's three legs does
   the fit supply, and which one is still a claim you would have to defend some other way?
4. **Whether you are done**, by the bar §5.3.2 set — and what the residual still contains
   that a serious model of this would have to treat.